# Gradient Accumulation & Activation Checkpointing

This notebook explores two complementary techniques for training large models under memory constraints:

1. **Gradient Accumulation** — simulate large batch sizes by accumulating gradients over multiple
   mini-batches before calling `optimizer.step()`
2. **Activation Checkpointing** (gradient checkpointing) — trade compute for memory by recomputing
   intermediate activations during the backward pass instead of storing them

We will:
1. Explain why large batch sizes matter and how gradient accumulation achieves them without extra memory
2. Show the math: why dividing the loss by the accumulation steps produces the correct gradient
3. Implement a gradient accumulation training loop from scratch
4. Explain what activation memory is and how checkpointing reduces it
5. Apply `torch.utils.checkpoint` to a ResNet-18
6. Benchmark all four configurations: baseline, accumulation only, checkpointing only, both combined

In [ ]:
import sys, os

# In Colab, clone the repo so local imports (src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/03_Training_Techniques")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
from torchvision import transforms
import torch.utils.checkpoint as cp
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from src.utils.device import set_device, set_seed, set_deterministic
from src.data.loaders import build_dataloaders_cifar

device = set_device()
set_seed(42)
set_deterministic()

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 1. Why Large Batch Sizes?

Larger batches give lower-variance gradient estimates, enabling larger learning rates and
faster convergence. Some methods (contrastive learning, distributed training) require very
large batches (4096+).

**Problem:** `batch_size * activation_memory` must fit in GPU RAM.

**Solution:** Accumulate gradients across multiple forward/backward passes, then step once.

| Actual batch size | Accumulation steps | Effective batch size | Activation memory |
|---|---|---|---|
| 128 | 1 | 128 | baseline |
| 32 | 4 | 128 | ~4x less |
| 16 | 8 | 128 | ~8x less |

## 2. The Math of Gradient Accumulation

The gradient of the loss over a full batch $B$ of size $N$:

$\nabla_\theta L_B = \frac{1}{N} \sum_{i=1}^{N} \nabla_\theta \ell(x_i, \theta)$

Split $B$ into $K$ mini-batches of size $n = N/K$:

$\nabla_\theta L_B = \frac{1}{K} \sum_{k=1}^{K} \nabla_\theta L_{B_k}$

So: compute each mini-batch loss $L_{B_k}$, divide by $K$, call `.backward()`. PyTorch
accumulates gradients by default (`.grad` is summed, not replaced), so after $K$ backward
passes the accumulated gradient equals the full-batch gradient.

**The key line:** `(loss / accumulation_steps).backward()`

In [ ]:
# Prove that gradient accumulation produces the same gradient as a single large batch
torch.manual_seed(0)
model_test = nn.Linear(4, 2, bias=False)
criterion_test = nn.MSELoss()

# Fixed data: 8 samples
X = torch.randn(8, 4)
y = torch.randn(8, 2)

# Method 1: Single forward pass with all 8 samples
model_test.zero_grad()
loss_full = criterion_test(model_test(X), y)
loss_full.backward()
grad_full = model_test.weight.grad.clone()

# Method 2: Accumulate over 4 mini-batches of 2 samples each
K = 4
model_test.zero_grad()
for k in range(K):
    X_k = X[k*2:(k+1)*2]
    y_k = y[k*2:(k+1)*2]
    loss_k = criterion_test(model_test(X_k), y_k)
    (loss_k / K).backward()  # divide by K before backward

grad_accum = model_test.weight.grad.clone()

max_diff = (grad_full - grad_accum).abs().max().item()
print(f"Max gradient difference: {max_diff:.2e}")
print(f"Gradients match: {torch.allclose(grad_full, grad_accum, atol=1e-6)}")

## 3. Gradient Accumulation Training Loop

The standard training loop changes in three ways:

```python
optimizer.zero_grad()                          # zero once at start
for i, (inputs, labels) in enumerate(dataloader):
    loss = criterion(model(inputs), labels)
    (loss / accumulation_steps).backward()     # 1. divide loss
    
    if (i + 1) % accumulation_steps == 0:      # 2. step every K batches
        optimizer.step()
        optimizer.zero_grad()                  # 3. zero after step

# Handle trailing batches at epoch end
if (i + 1) % accumulation_steps != 0:
    optimizer.step()
    optimizer.zero_grad()
```

Key points:
- `optimizer.zero_grad()` only at accumulation boundaries, not every batch
- `optimizer.step()` only every `accumulation_steps` iterations
- Always handle trailing batches (epoch length may not be divisible by `accumulation_steps`)

In [ ]:
from src.utils.device import get_amp_config


def train_model_grad_accum(model, train_dataloader, test_dataloader, epochs, criterion,
                           optimizer, device, accumulation_steps=1, use_amp=False,
                           scheduler=None):
    """Training loop with gradient accumulation and optional AMP."""
    model.to(device)

    amp_device_type, amp_dtype, use_scaler = None, None, False
    scaler = None
    if use_amp:
        amp_device_type, amp_dtype, use_scaler = get_amp_config(device)
        scaler = torch.GradScaler(device) if use_scaler else None

    results = {
        "train_loss_per_epoch": [], "train_acc_per_epoch": [],
        "val_loss_per_epoch": [], "val_acc_per_epoch": [], "epoch_times": [],
    }

    for epoch in tqdm(range(epochs)):
        model.train()
        training_loss, training_acc = 0.0, 0.0
        epoch_start = time.perf_counter()
        optimizer.zero_grad()

        for i, (inputs, labels) in enumerate(train_dataloader):
            inputs, labels = inputs.to(device), labels.to(device)

            if use_amp:
                with torch.autocast(device_type=amp_device_type, dtype=amp_dtype):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaled_loss = loss / accumulation_steps

            if scaler is not None:
                scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

            # Step every accumulation_steps batches
            if (i + 1) % accumulation_steps == 0:
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()

            training_loss += loss.item()
            training_acc += (outputs.argmax(1) == labels).float().mean().item()

        # Handle trailing batches
        if (i + 1) % accumulation_steps != 0:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        if scheduler is not None:
            scheduler.step()

        epoch_time = time.perf_counter() - epoch_start
        epoch_loss_tr = training_loss / len(train_dataloader)
        epoch_acc_tr = training_acc / len(train_dataloader)

        model.eval()
        test_loss, test_acc = 0.0, 0.0
        with torch.no_grad():
            for batch, (inputs, labels) in enumerate(test_dataloader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                test_acc += (outputs.argmax(1) == labels).float().mean().item()

        epoch_loss_te = test_loss / len(test_dataloader)
        epoch_acc_te = test_acc / len(test_dataloader)

        print(f"Epoch {epoch}: train_loss={epoch_loss_tr:.4f}, train_acc={epoch_acc_tr:.4f}, "
              f"val_loss={epoch_loss_te:.4f}, val_acc={epoch_acc_te:.4f}, time={epoch_time:.2f}s")

        results["train_loss_per_epoch"].append(epoch_loss_tr)
        results["train_acc_per_epoch"].append(epoch_acc_tr)
        results["val_loss_per_epoch"].append(epoch_loss_te)
        results["val_acc_per_epoch"].append(epoch_acc_te)
        results["epoch_times"].append(epoch_time)

    return model, results


print("Training function ready.")

## 4. What is Activation Memory?

During the forward pass, every layer stores its input (and sometimes intermediate values)
for use in the backward pass. For a network with $L$ layers:

```
Forward:  x --> [Layer 1] --> a1 --> [Layer 2] --> a2 --> ... --> [Layer L] --> loss
                              |                    |                           |
Stored:                      a1                   a2              ...         aL
                              |                    |                           |
Backward:              dL/dw1 <-- a1        dL/dw2 <-- a2              dL/dloss
```

**Activation memory** is $O(L \times B \times d)$ where $B$ is batch size and $d$ is
feature dimension. For large models, activation memory often **dominates** total memory —
larger than parameter + optimizer memory combined.

## 5. Activation Checkpointing

Instead of storing all $L$ activations, only store activations at "checkpoint" boundaries.
During backward, recompute the rest by re-running the forward pass for those layers.

```
Without checkpointing (all activations stored):
  [L1]->a1->[L2]->a2->[L3]->a3->[L4]->a4->[L5]->a5->[L6]->a6-> loss
  Stored: a1, a2, a3, a4, a5, a6  (6 activations)

With checkpointing at L1, L3, L5:
  [L1]->a1->[L2]->  ->[L3]->a3->[L4]->  ->[L5]->a5->[L6]->  -> loss
  Stored: a1, a3, a5  (3 activations)
  Backward: recompute a2 from a1, a4 from a3, a6 from a5
```

**Tradeoff:** ~30% more compute for ~50-70% less activation memory.

**Important:** Checkpointing saves only activation memory, NOT parameter or optimizer memory.

**PyTorch API:**
- `torch.utils.checkpoint.checkpoint(fn, *args, use_reentrant=False)` — wraps any callable
- `torch.utils.checkpoint.checkpoint_sequential(functions, segments, input)` — wraps `nn.Sequential`
- For HuggingFace models: `model.gradient_checkpointing_enable()` (HF-specific)

In [ ]:
# Demonstrate torch.utils.checkpoint: same output, same gradients, less memory
torch.manual_seed(0)

layer1 = nn.Linear(8, 8)
layer2 = nn.Linear(8, 4)

def two_layers(x):
    return layer2(F.relu(layer1(x)))

x = torch.randn(4, 8, requires_grad=True)

# Without checkpointing
out_normal = two_layers(x)
out_normal.sum().backward()
grad_normal = x.grad.clone()
x.grad = None

# With checkpointing
out_ckpt = cp.checkpoint(two_layers, x, use_reentrant=False)
out_ckpt.sum().backward()
grad_ckpt = x.grad.clone()

print(f"Output match: {torch.allclose(out_normal, out_ckpt)}")
print(f"Gradient match: {torch.allclose(grad_normal, grad_ckpt)}")
print(f"Max output diff: {(out_normal - out_ckpt).abs().max().item():.2e}")
print(f"Max gradient diff: {(grad_normal - grad_ckpt).abs().max().item():.2e}")

In [ ]:
def make_resnet18_cifar10():
    """Create a ResNet-18 adapted for CIFAR-10 (32x32 images, 10 classes)."""
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


class CheckpointedResNet18(nn.Module):
    """ResNet-18 with activation checkpointing on layer1-4."""
    def __init__(self):
        super().__init__()
        base = make_resnet18_cifar10()
        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool  # Identity for CIFAR
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.avgpool = base.avgpool
        self.fc = base.fc

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        # Checkpoint each residual layer block
        x = cp.checkpoint(self.layer1, x, use_reentrant=False)
        x = cp.checkpoint(self.layer2, x, use_reentrant=False)
        x = cp.checkpoint(self.layer3, x, use_reentrant=False)
        x = cp.checkpoint(self.layer4, x, use_reentrant=False)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# Verify both models produce correct output and have same param count
m1 = make_resnet18_cifar10()
m2 = CheckpointedResNet18()
p1 = sum(p.numel() for p in m1.parameters())
p2 = sum(p.numel() for p in m2.parameters())
print(f"Standard ResNet-18:      {p1:,} params")
print(f"Checkpointed ResNet-18:  {p2:,} params")
print(f"Parameter count matches: {p1 == p2}")
del m1, m2

## 6. Combining with AMP

Both techniques work seamlessly with mixed precision:
- **Gradient accumulation + AMP:** wrap forward in `autocast`, scale the divided loss
- **Activation checkpointing + AMP:** `autocast` context propagates into recomputed forward segments automatically (PyTorch 2.0+)

## 7. Benchmark Setup

We compare 4 configurations, all with the **same effective batch size** (128):

| Config | Batch size | Accum steps | Effective batch | Checkpointing |
|---|---|---|---|---|
| 1. Baseline | 128 | 1 | 128 | No |
| 2. Gradient accumulation | 32 | 4 | 128 | No |
| 3. Activation checkpointing | 128 | 1 | 128 | Yes |
| 4. Both combined | 32 | 4 | 128 | Yes |

Same effective batch size means accuracy should be comparable across all configs.

In [ ]:
NUM_EPOCHS = 5
LEARNING_RATE = 0.001
NUM_WORKERS = 4
ACCUMULATION_STEPS = 4

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

# Two dataloaders with different batch sizes
train_dl_128, test_dl, _, class_names = build_dataloaders_cifar(
    '../../data', train_transforms, test_transforms, NUM_WORKERS, batch_size=128)
train_dl_32, _, _, _ = build_dataloaders_cifar(
    '../../data', train_transforms, test_transforms, NUM_WORKERS, batch_size=32)

print(f"Batches (bs=128): {len(train_dl_128)}, Batches (bs=32): {len(train_dl_32)}")
print(f"Effective batch size: 32 x {ACCUMULATION_STEPS} = {32 * ACCUMULATION_STEPS}")

In [ ]:
print("=" * 60)
print("Config 1: Baseline (bs=128, no accumulation, no checkpointing)")
print("=" * 60)
set_seed(42)
model_1 = make_resnet18_cifar10()
opt_1 = torch.optim.Adam(model_1.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
model_1, results_1 = train_model_grad_accum(
    model_1, train_dl_128, test_dl, NUM_EPOCHS, criterion, opt_1, device,
    accumulation_steps=1)

In [ ]:
print("=" * 60)
print("Config 2: Gradient Accumulation (bs=32, accum_steps=4)")
print("=" * 60)
set_seed(42)
model_2 = make_resnet18_cifar10()
opt_2 = torch.optim.Adam(model_2.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
model_2, results_2 = train_model_grad_accum(
    model_2, train_dl_32, test_dl, NUM_EPOCHS, criterion, opt_2, device,
    accumulation_steps=ACCUMULATION_STEPS)

In [ ]:
print("=" * 60)
print("Config 3: Activation Checkpointing (bs=128)")
print("=" * 60)
set_seed(42)
model_3 = CheckpointedResNet18()
opt_3 = torch.optim.Adam(model_3.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
model_3, results_3 = train_model_grad_accum(
    model_3, train_dl_128, test_dl, NUM_EPOCHS, criterion, opt_3, device,
    accumulation_steps=1)

In [ ]:
print("=" * 60)
print("Config 4: Both (bs=32, accum_steps=4, checkpointed)")
print("=" * 60)
set_seed(42)
model_4 = CheckpointedResNet18()
opt_4 = torch.optim.Adam(model_4.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
model_4, results_4 = train_model_grad_accum(
    model_4, train_dl_32, test_dl, NUM_EPOCHS, criterion, opt_4, device,
    accumulation_steps=ACCUMULATION_STEPS)

## 8. Results

In [ ]:
configs = [
    ('Baseline', results_1),
    ('Grad Accum', results_2),
    ('Checkpointing', results_3),
    ('Both', results_4),
]

t_baseline = sum(results_1['epoch_times'])
rows = []
for name, r in configs:
    total = sum(r['epoch_times'])
    rows.append({
        'Config': name,
        'Total time (s)': f"{total:.1f}",
        'Avg epoch (s)': f"{total/NUM_EPOCHS:.1f}",
        'Final val acc': f"{r['val_acc_per_epoch'][-1]:.4f}",
        'Final val loss': f"{r['val_loss_per_epoch'][-1]:.4f}",
        'Speedup': f"{t_baseline/total:.2f}x",
    })

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
labels = ['Baseline', 'Grad Accum', 'Checkpointing', 'Both']
all_results = [results_1, results_2, results_3, results_4]

# Plot 1: Epoch training times
ax = axes[0]
x = np.arange(NUM_EPOCHS)
width = 0.2
for j, (r, label, color) in enumerate(zip(all_results, labels, colors)):
    ax.bar(x + j * width - 1.5 * width, r['epoch_times'], width,
           label=label, color=color, alpha=0.7)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Training Time per Epoch', fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Validation accuracy
ax = axes[1]
for r, label, color in zip(all_results, labels, colors):
    ax.plot(range(NUM_EPOCHS), r['val_acc_per_epoch'], 'o-', label=label,
            linewidth=2, color=color)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Accuracy Convergence', fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 3: Training loss
ax = axes[2]
for r, label, color in zip(all_results, labels, colors):
    ax.plot(range(NUM_EPOCHS), r['train_loss_per_epoch'], 'o-', label=label,
            linewidth=2, color=color)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Loss Convergence', fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Memory Analysis

ResNet-18 memory breakdown (theoretical):

| Component | Baseline (bs=128) | Accum (bs=32) | Ckpt (bs=128) | Both (bs=32) |
|---|---|---|---|---|
| Parameters | 44.7 MB | 44.7 MB | 44.7 MB | 44.7 MB |
| Optimizer (Adam) | 89.4 MB | 89.4 MB | 89.4 MB | 89.4 MB |
| Activations (est.) | ~300 MB | ~75 MB | ~100 MB | ~25 MB |
| **Total (est.)** | **~434 MB** | **~209 MB** | **~234 MB** | **~159 MB** |

- **Gradient accumulation** reduces activation memory proportional to the batch size reduction (4x smaller batch → ~4x less activation memory)
- **Checkpointing** reduces activation memory by not storing intermediate activations (only checkpoint boundaries)
- **Both combined** gives the best memory savings
- **Parameter and optimizer memory are unchanged** by either technique

In [ ]:
def estimate_activation_memory(model, input_shape, device):
    """Estimate activation memory by hooking into forward pass."""
    activation_sizes = []
    hooks = []

    def hook_fn(module, input, output):
        if isinstance(output, torch.Tensor):
            activation_sizes.append(output.nelement() * output.element_size())

    for module in model.modules():
        if module is model:
            continue
        hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        model.to(device)
        model(torch.randn(*input_shape, device=device))

    for h in hooks:
        h.remove()

    return sum(activation_sizes)


# Compare activation memory for different configs
print("Estimated activation memory (single forward pass):")
for name, bs, model_fn in [
    ('Baseline (bs=128)', 128, make_resnet18_cifar10),
    ('Grad Accum (bs=32)', 32, make_resnet18_cifar10),
    ('Ckpt (bs=128)', 128, CheckpointedResNet18),
    ('Both (bs=32)', 32, CheckpointedResNet18),
]:
    m = model_fn()
    mem = estimate_activation_memory(m, (bs, 3, 32, 32), device)
    print(f"  {name:25s}: {mem / 1024**2:.1f} MB")
    del m

## 10. Reusable Implementation

`train_model_grad_accum` is available in `src/training/trainer.py`:

```python
from src.training.trainer import train_model_grad_accum
```

Activation checkpointing is model-level — wrap your model's forward pass with
`torch.utils.checkpoint.checkpoint()` as shown in the `CheckpointedResNet18` class.

In [ ]:
from src.training.trainer import train_model_grad_accum as accum_fn

print(f"train_model_grad_accum imported successfully.")
print("All src/ imports verified.")

## 11. Key Takeaways

1. **Gradient accumulation simulates large batch sizes.** Forward/backward multiple mini-batches, divide loss by $K$, step once. The gradient is mathematically identical to a single large batch.

2. **The key detail is loss normalization.** Always divide by `accumulation_steps` before `.backward()` so gradient magnitudes match the effective batch size.

3. **Activation checkpointing trades compute for memory.** Recompute intermediate activations during backward instead of storing them. ~30% more compute for ~50-70% less activation memory.

4. **Checkpointing saves only activation memory.** Parameter and optimizer memory are unchanged. Use gradient accumulation (smaller batch size) to further reduce activation memory.

5. **The two techniques are complementary.** Accumulation reduces per-step memory via smaller batches. Checkpointing reduces per-layer storage via recomputation. Combined, they enable training models that would otherwise not fit in memory.

6. **Accuracy is equivalent.** All four configurations converge similarly since they share the same effective batch size.

| Technique | What it saves | Tradeoff | PyTorch API |
|---|---|---|---|
| Gradient Accumulation | Activation memory (smaller batch) | More optimizer steps per epoch | Manual loop changes |
| Activation Checkpointing | Activation memory (fewer stored) | ~30% more compute | `torch.utils.checkpoint.checkpoint()` |

### When to Use Each

- **Gradient accumulation:** When you need a large effective batch size but can't fit it in memory. Common in distributed training, contrastive learning.
- **Activation checkpointing:** When the model has many layers with large activations. Common with LLMs, ViT-Large, deep ResNets.
- **Both:** When training very large models with large effective batch sizes (e.g., GPT pretraining).

### Further Reading

- Chen et al. (2016). *Training Deep Nets with Sublinear Memory Cost.* https://arxiv.org/abs/1604.06174
- PyTorch docs: `torch.utils.checkpoint` https://pytorch.org/docs/stable/checkpoint.html

## 12. Run on GPU (Modal)

Run all 4 benchmark configurations on a remote NVIDIA GPU via [Modal](https://modal.com).
On CUDA, you will see actual memory savings from gradient accumulation and checkpointing.

```bash
# One-time setup:
pip install modal
modal token set
```

In [ ]:
from src.infra.modal_runner import run_training

gpu_configs = [
    ("Baseline",       "make_resnet18_cifar10",  128, 1),
    ("Grad Accum",     "make_resnet18_cifar10",  32,  4),
    ("Checkpointing",  "CheckpointedResNet18",   128, 1),
    ("Both",           "CheckpointedResNet18",   32,  4),
]

gpu_results = {}
for name, model_name, bs, accum in gpu_configs:
    print(f"Running {name}...")
    gpu_results[name] = run_training.remote(
        model_factory_name=model_name,
        trainer_name="train_model_grad_accum",
        dataset="cifar10",
        epochs=5,
        batch_size=bs,
        lr=0.001,
        accumulation_steps=accum,
    )

print(f"
GPU: {gpu_results['Baseline']['gpu_name']}
")
for name, r in gpu_results.items():
    total = sum(r["epoch_times"])
    acc = r["val_acc_per_epoch"][-1]
    mem = r["peak_memory_mb"]
    print(f"{name:20s} | Time: {total:6.1f}s | Val acc: {acc:.4f} | Peak mem: {mem:.0f} MB")